## T6

In [68]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(42)

d) Генерация выборки для нек. значения параметра, вычисление доверительных интервалов для beta = 0.95

In [69]:
def InvPareto(x, th):
    return (1 - x)**(1 / (1 - th))

def med_func(th):
    return 2 ** (1/(th-1))

n = 100
beta = 0.95
#пусть
theta = 5
X = InvPareto(st.uniform(loc=0, scale=1).rvs(size=n), th=theta)
X

array([1.12447586, 2.12236357, 1.38983693, 1.25638134, 1.04331821,
       1.04331075, 1.01507215, 1.65335692, 1.25831124, 1.3604459 ,
       1.00521337, 2.40100973, 1.56300052, 1.06148822, 1.05144956,
       1.05195765, 1.09492781, 1.20440053, 1.15186719, 1.08986708,
       1.26692502, 1.03827287, 1.09021931, 1.1208298 , 1.1644323 ,
       1.4688562 , 1.0572635 , 1.19782485, 1.25154174, 1.01196194,
       1.2634338 , 1.04784986, 1.01695817, 2.10311998, 2.32253111,
       1.51147031, 1.09507403, 1.02602727, 1.33400766, 1.15606582,
       1.03307321, 1.18635645, 1.00878681, 1.82231145, 1.07773805,
       1.31201477, 1.09788615, 1.20144827, 1.21872588, 1.05242513,
       2.39456547, 1.45217342, 2.01631907, 1.75599913, 1.25578843,
       1.89147859, 1.02343423, 1.05604804, 1.01163769, 1.10338532,
       1.13092126, 1.08235596, 1.55447748, 1.11662054, 1.08594515,
       1.21604255, 1.03870478, 1.49948378, 1.01955776, 2.9551121 ,
       1.44754779, 1.05694731, 1.00138531, 1.52573185, 1.35903

In [70]:
#Асимптотический доверительный интервал для медианы
theta_omp = 1 + 1 / (np.mean(np.log(X)))
med = med_func(theta)
med_omp = med_func(theta_omp)
t1 = st.norm.ppf((1-beta)/2)
t2 = st.norm.ppf((1+beta)/2)
asymp_med_left = med_omp - med_omp*np.log(2)/(np.sqrt(n)*(theta_omp-1))*t2
asymp_med_right = med_omp - med_omp*np.log(2)/(np.sqrt(n)*(theta_omp-1))*t1
asymp_med_l = asymp_med_right - asymp_med_left
print(f"Медиана: {med}, oценка медианы:{med_omp}")
print("Асимптотический доверительный интервал для медианы:", asymp_med_left, "< theta <", asymp_med_right)
print("l =", asymp_med_l)

Медиана: 1.189207115002721, oценка медианы:1.1717680430090054
Асимптотический доверительный интервал для медианы: 1.1353634066375606 < theta < 1.2081726793804501
l = 0.07280927274288951


In [71]:
#Асимптотический доверительный интервал для theta
t1 = st.norm.ppf((1-beta)/2)
t2 = st.norm.ppf((1+beta)/2)
asymp_left = theta_omp - t2 / (np.sum(np.log(X))/np.sqrt(n))
asymp_right = theta_omp - t1 / (np.sum(np.log(X))/np.sqrt(n))
asymp_l = asymp_right - asymp_left
print(f"theta: {theta}, oценка theta ОМП:{theta_omp}")
print("Асимптотический доверительный интервал для theta:", asymp_left, "< theta <", asymp_right)
print("l =", asymp_l)

theta: 5, oценка theta ОМП:5.3727888219392215
Асимптотический доверительный интервал для theta: 4.515737961639201 < theta < 6.229839682239242
l = 1.7141017206000413


In [72]:
#Непараметрический бутстрап
N = 1000

def bootstrap_np_OMP(x, N): #np - non parametric
    arr = []
    n = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)        
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]
    return sorted(np.array(arr))


bs_arr = bootstrap_np_OMP(X, N)
t1 = int((1 - beta)/2 * N - 1) #т.к. индексация с 0 в массиве
t2 = int((1 + beta)/2 * N - 1)
bs_np_left = theta_omp - bs_arr[t2]
bs_np_right = theta_omp - bs_arr[t1]
bs_np_l = bs_np_right - bs_np_left
print("Непараметрический бутстраповский доверительный интервал для theta(ОМП):", bs_np_left, "< theta <", bs_np_right)
print("l =", bs_np_l)

Непараметрический бутстраповский доверительный интервал для theta(ОМП): 4.002568036702308 < theta < 5.706333453383885
l = 1.703765416681577


In [73]:
#Параметрический бутстрап
N = 50000

def bootstrap_p_OMP(x, N): #p - parametric
    arr = []
    n = len(x)
    for _ in range(N):
        sample = InvPareto(st.uniform(loc=0, scale=1).rvs(size=n), th=theta_omp)        
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]
    return sorted(np.array(arr))


bs_arr = bootstrap_np_OMP(X, N)
t1 = int((1 - beta)/2 * N - 1) #т.к. индексация с 0 в массиве
t2 = int((1 + beta)/2 * N - 1)
bs_p_left = theta_omp - bs_arr[t2]
bs_p_right = theta_omp - bs_arr[t1]
bs_p_l = bs_p_right - bs_p_left
print("Параметрический бутстраповский доверительный интервал для theta(ОМП):", bs_p_left, "< theta <", bs_p_right)
print("l =", bs_p_l)

Параметрический бутстраповский доверительный интервал для theta(ОМП): 3.976693252182283 < theta < 5.735873232497911
l = 1.7591799803156283


f) Сравнить полученные доверительные интервалы

In [74]:
sort_lens = sorted([(asymp_l, "Асимптотический"), (bs_np_l, "Бутстрап непараметрический"), (bs_p_l, "Бутстрап параметрический")])
print("Рейтинг доверительных интервалов (для theta):")
for i in range(len(sort_lens)):
    print(f"{i+1})", sort_lens[i][1], f"(l = {np.round(sort_lens[i][0],3)})")

Рейтинг доверительных интервалов (для theta):
1) Бутстрап непараметрический (l = 1.704)
2) Асимптотический (l = 1.714)
3) Бутстрап параметрический (l = 1.759)
